In [0]:
import pandas as pd
# import matplotlib.pyplot as plt
from scipy import stats
import numpy as np
np.random.seed(42)

# confidence = 0.80       # 0.95 for 95 % Confidence Interval
# test_side = 2           # 1 for one sided, 2 for two sided
# relative_mean = 'Lower' # Claim of relative position of population mean, 'Lower' or 'Higher', needed for 1 sided test

confidence = eval(dbutils.widgets.get("confidence"))
test_side = eval(dbutils.widgets.get("test_side"))
relative_mean = (dbutils.widgets.get("ts_relative_mean"))
alpha = 1 - confidence
prob = 1 - alpha / test_side

def print_stats(test_name, distribution, score, value, lower, upper):
    if distribution == 't':
        print(f"t-score: {score:.5f} for t({prob:.4f}, {df:.2f})")
        print(f"C.I. for {test_name}: ({lower:.5f}, {upper:.5f})")
        print(f"t-value: {value:.5f}")

    if distribution == 'Z':
        print(f"Z-score: {score:.5f} for Z({prob:.4f})")
        print(f"C.I. for {test_name}: ({lower:.5f}, {upper:.5f})")
        print(f"Z-value: {value:.5f}")
    
    print(f"p-value: {p_value* 100:.2f} %")
    if p_value < alpha:
        print("Reject the Null Hypothesis")
    else:
        print("Accept the Null Hypothesis")

def test_output(distribution, score, value, factor, var):
    if distribution == 't':
        t_score, t_value, sigma_factor = score, value, factor
        if test_side == 2:  
            p_value = 2 * (1 - stats.t.cdf(abs(t_value), df))
            CI_lower = var - t_score * sigma_factor
            CI_upper = var + t_score * sigma_factor
        else:
            if relative_mean == "Lower": 
                p_value = stats.t.cdf(t_value, df)
                CI_lower = -np.inf
                CI_upper = var + t_score * sigma_factor
            else:  
                p_value = 1 - stats.t.cdf(t_value, df)
                CI_lower = var - t_score * sigma_factor
                CI_upper = np.inf

    if distribution == 'Z':
        Z_score, Z_value, sigma_factor = score, value, factor
        if test_side == 2:   
            p_value = 2 * (1 - stats.norm.cdf(abs(Z_value)))
            CI_lower = var - Z_score * sigma_factor
            CI_upper = var + Z_score * sigma_factor
        else:
            if relative_mean == "Lower": 
                p_value = stats.norm.cdf(Z_value)
                CI_lower = -np.inf
                CI_upper = var + Z_score * sigma_factor
            else: 
                p_value = 1 - stats.norm.cdf(Z_value)
                CI_lower = var - Z_score * sigma_factor
                CI_upper = np.inf
    return p_value, CI_lower, CI_upper

In [0]:
def test_output2(distribution, score, value, factor, var):
    if distribution == 't':
      integral = stats.t.cdf(abs(value), df)
    else:
      integral = stats.norm.cdf(abs(value))

    if test_side == 2:
        p_value = 2 * (1 - integral)
        CI_lower =var - score * factor
        CI_upper = var + score * factor
    else:
        if relative_mean == "Lower": 
            p_value = integral
            CI_lower = -np.inf
            CI_upper = var + score * factor
        else:  
            p_value = 1 - integral
            CI_lower = var - score * factor
            CI_upper = np.inf

    return p_value, CI_lower, CI_upper

# Measure Central Tendency

**Mean Median Mode Range IQR Variance Standard Dev**

In [0]:
# Insert Sample Data
data = [10, 20, 50, 40, 50, 60, 70, 80]

In [0]:
# Insert Sample Data
s = pd.Series(data)

# 8. Visualizations
plt.figure(figsize=(6, 3))

# Subplot 1: Histogram
plt.subplot(1, 2, 1)
plt.hist(s, color='skyblue', edgecolor='black', alpha=0.7)
plt.title("Distribution of Data")
plt.xlabel("Value")
plt.ylabel("Frequency")

# Plot 2: Box Plot
plt.subplot(1, 2, 2)
plt.boxplot(s)
plt.title("Box Plot of Data")
plt.ylabel("Values")

plt.tight_layout()
plt.show()

In [0]:
# 1. Mean (Average)
mean_val = s.mean()

# 2. Median (Middle value)
median_val = s.median()

# 3. Mode (Most frequent value)
# Note: Returns a Series (use .iloc[0] for the first mode)
mode_val = s.mode().iloc[0]

# 4. Range (Max - Min)
data_range = s.max() - s.min()

# 5. IQR (Interquartile Range: Q3 - Q1)
iqr_val = s.quantile(0.75) - s.quantile(0.25)

# 6. Variance
variance_val = s.var()

# 7. Standard Deviation
std_dev_val = s.std()

# Print results
print("Sample Statistics:\n")
print(f"Count:{len(s)},  Mean: {mean_val}, Std Dev: {std_dev_val:.5f}, Variance: {variance_val:.5f}")
print(f"Median: {median_val}, Range: {data_range}, IQR: {iqr_val}, Mode: {mode_val}")

# Type of tests

|Type | Condition | Test     | Reasoning           |
|---| ------------------------------------------------------------------------------------------ | --------------------------------- | ------------------------------------------------------------------------------ |
| A/B Test Control vs Target, A/B test Promo A vs Promo B| Two independent groups comparing **average metric** (revenue, time spent)                  | **Independent Two-Sample t-test** | Tests whether the **means of two unrelated groups differ**.                    |
|Same patients before vs after |Same subjects measured **before and after** (e.g., spend before vs after promotion)        | **Paired t-test**                 | Tests whether the **mean difference within the same subjects** is significant. |
|A/B test of proportions | Two groups comparing **conversion rate (binary outcome)**                                  | **Two-Proportion Z-test**         | Compares whether **two population proportions differ**.                        |
|2 Brands: converted vs not converted, Hotel reviews |Comparing **frequency counts across categories** (brand conversions, ratings distribution) | **Chi-Square Test**, **Chi-Square Test of Independance**               | Tests whether **categorical distributions are independent**.                   |
| Data **not normally distributed** or contains strong outliers                            |  | **Mann–Whitney U Test**           | Non-parametric test comparing **median ranks of two independent groups**.      |
| Comparing **3 or more groups** (Promo A, B, C) on a continuous metric                     | | **ANOVA**                         | Tests whether **at least one group mean differs** among multiple groups.       |


# One Sample test for Mean

In [0]:
# sample stats insert here
n = 100                    # sample size
X_bar = 52                 # sample mean
sample_std = None          # sample std dev (needed if sigma is unknown)

# Insert Population Parameters
μ0 = 50                # Hypothesized poulation mean, None if not known
sigma = 10             # population standard deviation, None if not known

n, μ0, X_bar, sigma, sample_std = 25, 168, 172.50, 15.40, None

In [0]:
# Compute the Confidence Interval for Population Mean
if sigma is None:
    # ---- T-interval (sigma unknown) ----
    df = n - 1
    t_score = stats.t.ppf(prob, df)
    sigma_factor = sample_std / np.sqrt(n)
    t_value = (X_bar - (μ0)) /  (sample_std / np.sqrt(n))

    print(f"Sample Mean: {X_bar}")
    print(f"Sample Std Dev: {sample_std}")

    p_value, CI_lower, CI_upper = test_output2('t', t_score, t_value, sigma_factor, X_bar)
    print_stats('Mean', 't', t_score, t_value, CI_lower, CI_upper)

else:
    # ---- Z-interval (sigma known) ----
    z_score = stats.norm.ppf(prob)
    sigma_factor = sigma / np.sqrt(n)
    Z_value = (X_bar - (μ0)) /  (sigma / np.sqrt(n))

    print(f"Sample Mean: {X_bar}")
    print(f"Population Std Dev: {sigma}")

    p_value, CI_lower, CI_upper = test_output2('Z', z_score, Z_value, sigma_factor, X_bar)
    print_stats('Mean', 'Z', z_score, Z_value, CI_lower, CI_upper)


In [0]:
# Compute the Confidence Interval for Population Mean
if sigma is None:
    # ---- T-interval (sigma unknown) ----
    df = n - 1
    t_score = stats.t.ppf(prob, df)
    margin_error = t_score * sample_std / np.sqrt(n)
    t_value = (X_bar - (μ0)) /  (sample_std / np.sqrt(n))

    print(f"T-score: {t_score:.5f} at T({prob:.4f}, {df})")
    print(f"Sample Mean: {X_bar}")
    print(f"Sample Std Dev: {sample_std}")

else:
    # ---- Z-interval (sigma known) ----
    z_score = stats.norm.ppf(prob)
    margin_error = z_score * sigma / np.sqrt(n)

    print(f"Z-score: {z_score:.5f} at Z({prob:.4f})")
    print(f"Sample Mean: {X_bar}")
    print(f"Population Std Dev: {sigma}")


ci_lower, ci_upper  = X_bar - margin_error, X_bar + margin_error

if test_side == 2:
    print(f"{int(confidence*100)}% CI for Population Mean: ({ci_lower:.5f}, {ci_upper:.5f})")
else:
    print(f"Right-tailed CI ({μ0} < μ): [{ci_lower:.4f}, ∞)")
    print(f"Left-tailed CI (μ < {μ0}): (-∞, {ci_upper:.4f}])")


In [0]:
# Compare if Population mean is within the confidence interval

if test_side == 2:
    if ci_lower <= μ0 <= ci_upper:
        print(f"Population mean {μ0} is within the confidence interval.")
        print("Accept the Null Hypothesis")
    else:
        print(f"Population mean {μ0} is NOT within the confidence interval.")
        print("Reject the Null Hypothesis")
else:
    if relative_mean == 'Lower':
        if ci_upper >= μ0:
            print(f"Population mean {μ0} is within the confidence interval.")
            print("Accept the Null Hypothesis")
        else:
            print(f"Population mean {μ0} is NOT within the confidence interval.")
            print("Reject the Null Hypothesis")
    else:
        if ci_lower <= μ0:
            print(f"Population mean {μ0} is within the confidence interval.")
            print("Accept the Null Hypothesis")
        else:
            print(f"Population mean {μ0} is NOT within the confidence interval.")
            print("Reject the Null Hypothesis")
    


In [0]:
if sigma is None:
    t_value = (X_bar - (μ0)) /  (sample_std / np.sqrt(n))
    print(f"T-value: {t_value:.5f}")
    if test_side == 2:
        p_value = 2 * (1 - stats.t.cdf(abs(t_value), df))
    elif test_side != 2 and relative_mean == 'Lower':
        p_value = p_value = stats.t.cdf(t_value, df)
    else:
        p_value = 1 - stats.t.cdf(t_value, df)
else:
    Z_value = (X_bar - (μ0)) /  (sigma / np.sqrt(n))
    print(f"Z-value: {Z_value:.5f}")
    if test_side == 2:
        p_value = 2 * (1 - stats.norm.cdf(abs(Z_value)))
    elif test_side != 2 and relative_mean == 'Lower':
        p_value = stats.norm.cdf(Z_value)
    else:
        p_value = 1 - stats.norm.cdf(Z_value)

print(f"p-value: {p_value:.5f}")
if p_value < alpha:
    print("Reject the Null Hypothesis")
else:
    print("Accept the Null Hypothesis")

**Determine Sample Size**

# Two Sample Tests for ΔMean

### Independant Samples

In [0]:
# Sample sizes
n1, n2 = 20, 20

# Sample means, provide one of the following
X1, X2 = 3.27, 2.53
# Xd = X2 - X1

# Hypothesized Population Means, provide one of the following
μ1, μ2 = 0, 0
# μd = μ2 - μ1

# Population Variances known
sigma1, sigma2 = 1, 1

Z_score = stats.norm.ppf(prob)
sigma_factor = np.sqrt(sigma1**2/n1 + sigma2**2/n2)
Z_value = (X2 - X1 - (μ2 - μ1)) / sigma_factor

#     if relative_mean == "Lower":  # HA: μ2 - μ1 < 0
#     else:                         # HA: μ2 - μ1 > 0

p_value, CI_lower, CI_upper = test_output2('Z', Z_score, Z_value, sigma_factor, X2 - X1)

print_stats('Difference in Means', 'Z', Z_score, Z_value, CI_lower, CI_upper)


In [0]:
# Sample sizes
n1, n2 = 21, 25

# Sample means, provide one of the following
X1, X2 = 3.27, 2.53
# Xd = X2 - X1

# Hypothesized Population Means, provide one of the following
μ1, μ2 = 0, 0
# μd = μ2 - μ1

# Sample variances for Population Variances not known and assumed equal
s1, s2 = 1.30, 1.16
sp = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))

# Degrees of Freedom
df = n1 + n2 - 2

t_score = stats.t.ppf(prob, df)
sigma_factor = np.sqrt(sp**2/n1 + sp**2/n2)
t_value = (X2 - X1 - (μ2 - μ1)) / sigma_factor

#     if relative_mean == "Lower":   # H₁: μ₂ − μ₁ < 0
#     else:                          # H₁: μ₂ − μ₁ > 0

p_value, CI_lower, CI_upper = test_output2('t', t_score, t_value, sigma_factor, X2 - X1)

print_stats('Difference in Means', 't', t_score, t_value, CI_lower, CI_upper)

In [0]:
# Sample sizes
n1, n2 = 20, 20

# Sample means, provide one of the following
X1, X2 = 3.27, 2.53
# Xd = X2 - X1

# Hypothesized Population Means, provide one of the following
μ1, μ2 = 0, 0
# μd = μ2 - μ1

# Sample variances for Population Variances not known and not assumed equal
s1, s2 = 1.30, 1.16

# Degrees of Freedom
# Welch's T test
df = (s1**2/n1 + s2**2/n2)**2 / ((s1**2/n1)**2/(n1 - 1) + (s2**2/n2)**2/(n2 - 1))

t_score = stats.t.ppf(prob, df)
sigma_factor = np.sqrt(s1**2/n1 + s2**2/n2)
t_value = (X2 - X1 - (μ2 - μ1)) / sigma_factor

#     if relative_mean == "Lower":   # H₁: μ₂ − μ₁ < 0
#     else:                          # H₁: μ₂ − μ₁ > 0

p_value, CI_lower, CI_upper = test_output2('t', t_score, t_value, sigma_factor, X2 - X1)

print_stats('Difference in Means', 't', t_score, t_value, CI_lower, CI_upper)

### Related Sample



---

**Choosing Between Independent and Related Samples**

The choice depends entirely on the **experimental design**:

* **Use Independent Samples** when comparing **two separate populations**.
  *Example:*
  *“Do people in New York spend more than people in London?”*

* **Use Related (Paired) Samples** when measuring an **effect, change, or difference within the same group** over time or conditions.
  *Example:*
  *“Did this training program increase employee speed?”*

---

**Key Differences**

| Feature          | Independent Samples                 | Related (Paired) Samples          |
| ---------------- | ----------------------------------- | --------------------------------- |
| Subjects         | Different individuals in each group | Same individuals or matched pairs |
| Relationship     | No natural pairing                  | Observations are linked           |
| Sample Size      | Groups may differ in size           | Groups must be equal in size      |
| What is compared | Means of two groups                 | Mean of within-pair differences   |



In [0]:
# sample stats insert here
n = 5                     # paired sample size
D_bar = -4.2              # sample mean differences
sample_std = 5.67         # sample std dev of differences (needed if sigma is unknown)

# Insert Population Parameters
μ0 = 0                  # Hypothesized poulation mean difference, None if not known
sigma = None            # population standard deviation of differences, None if not known

In [0]:
# Compute the Confidence Interval for Population Mean
if sigma is None:
    # ---- T-interval (sigma unknown) ----
    df = n - 1
    t_score = stats.t.ppf(prob, df)
    margin_error = t_score * sample_std / np.sqrt(n)

    print(f"T-score: {t_score:.5f} at T({prob:.4f}, {df})")
    print(f"Sample Mean Diff: {D_bar}")
    print(f"Sample Std Dev of Diff: {sample_std}")

else:
    # ---- Z-interval (sigma known) ----
    z_score = stats.norm.ppf(prob)
    margin_error = z_score * sigma / np.sqrt(n)

    print(f"Z-score: {z_score:.5f} at Z({prob:.4f})")
    print(f"Sample Mean Diff: {D_bar}")
    print(f"Population Std Dev of Diff: {sigma}")


ci_lower, ci_upper  = D_bar - margin_error, D_bar + margin_error

if test_side == 2:
    print(f"{int(confidence*100)}% CI for Population Mean: ({ci_lower:.5f}, {ci_upper:.5f})")
else:
    print(f"Right-tailed CI ({μ0} < μ): [{ci_lower:.4f}, ∞)")
    print(f"Left-tailed CI (μ < {μ0}): (-∞, {ci_upper:.4f}])")


In [0]:
# Compare if Population mean is within the confidence interval

if test_side == 2:
    if ci_lower <= μ0 <= ci_upper:
        print(f"Population mean {μ0} is within the confidence interval.")
        print("Accept the Null Hypothesis")
    else:
        print(f"Population mean {μ0} is NOT within the confidence interval.")
        print("Reject the Null Hypothesis")
else:
    if relative_mean == 'Lower':
        if ci_upper >= μ0:
            print(f"Population mean {μ0} is within the confidence interval.")
            print("Accept the Null Hypothesis")
        else:
            print(f"Population mean {μ0} is NOT within the confidence interval.")
            print("Reject the Null Hypothesis")
    else:
        if ci_lower <= μ0:
            print(f"Population mean {μ0} is within the confidence interval.")
            print("Accept the Null Hypothesis")
        else:
            print(f"Population mean {μ0} is NOT within the confidence interval.")
            print("Reject the Null Hypothesis")
    


In [0]:
if sigma is None:
    t_value = (D_bar - (μ0)) /  (sample_std / np.sqrt(n))
    print(f"T-value: {t_value:.5f}")
    if test_side == 2:
        p_value = 2 * (1 - stats.t.cdf(abs(t_value), df))
    elif test_side != 2 and relative_mean == 'Lower':
        p_value = p_value = stats.t.cdf(t_value, df)
    else:
        p_value = 1 - stats.t.cdf(t_value, df)
else:
    Z_value = (D_bar - (μ0)) /  (sigma / np.sqrt(n))
    print(f"Z-value: {Z_value:.5f}")
    if test_side == 2:
        p_value = 2 * (1 - stats.norm.cdf(abs(Z_value)))
    elif test_side != 2 and relative_mean == 'Lower':
        p_value = stats.norm.cdf(Z_value)
    else:
        p_value = 1 - stats.norm.cdf(Z_value)

print(f"p-value: {p_value:.5f}")
if p_value < alpha:
    print("Reject the Null Hypothesis")
else:
    print("Accept the Null Hypothesis")

# One Sample Test for Proportion

In [0]:
# sample stats insert here
n = 500                 # sample size
p = 0.05                # sample proportion
X = None                # number of successes

π = 0.08                # hypothesized population proportion

In [0]:
if p == None and X != None:
    p = X / n
if π == None:
  π = p
  
sigma_p = np.sqrt(π * (1 - π) / n) # population std deviation

if (n*π >= 5 and n*(1-π) >= 5):
  print(
        f"Normal approximation is valid with mean = {π:.3f} "
        f"\nand standard deviation = {sigma_p:.5f}"
    )
else:
  print("Cannot be approximated by a normal distribution")

In [0]:
# Standard Error
SE = np.sqrt(π * (1 - π) / n)

# Z-score
z_score = stats.norm.ppf(prob)
z_value = (p - π) / SE

    # if relative_mean == "Lower":   # H₁: p < π
    # else:                          # H₁: p > π

p_value, CI_lower, CI_upper = test_output2('Z', z_score, z_value, SE, p)

print_stats('Proportion', 'Z', z_score, z_value, CI_lower, CI_upper)

# Two Sample Tests for ΔProportions

In [0]:
n1, n2 = 72, 50                     # sample sizes
X1, X2 = 36, 31                     # number of successes
p1, p2 = X1/n1, X2/n2               # sample proportion

π1, π2 = 0, 0                       # hypothesized population proportion

In [0]:
p_bar = (X1 + X2) / (n1 + n2)

# Standard Error
SE = np.sqrt(p_bar * (1 - p_bar) * (1 / n1 + 1 / n2))

# Z-score
z_score = stats.norm.ppf(prob)
z_value = ((p1 - p2) - (π1 - π2))/ SE

# Margin Error
ME_factor = np.sqrt((p1 * (1- p1) / n1) + (p2 * (1- p2) / n2))

#     if relative_mean == 'Lower':            # HA: π2 − π1 < 0     
#     else:                                   # HA: π2 − π1 > 0

p_value, CI_lower, CI_upper = test_output2('Z', z_score, z_value, ME_factor, p1 - p2)

print_stats('Proportion Difference', 'Z', z_score, z_value, CI_lower, CI_upper)

# Two Sample test for ΔVariance 

In [0]:
# Sample sizes
n1, n2 = 21, 25

# Sample variances for Population Variances not known and assumed equal
# H0: sigma1 = sigma2
# HA: sigma1 != sigma2
s1, s2 = 1.30, 1.16

F_lower = stats.f.ppf(1 - prob, n1 - 1, n2 - 1)
F_upper = stats.f.ppf(prob, n1 - 1, n2 - 1)
F_value = s1**2 / s2**2
print(f"F-value: {F_value:.5f}")


if test_side == 2:
    print(f"F-C.I.: ({F_lower:.5f}, {F_upper:.5f})")
    if F_value < F_lower or F_value > F_upper:
        print("Reject the Null Hypothesis: Both Variances are NOT Equal")
    else:
        print("Accept the Null Hypothesis: Both Variances are Equal")
else:
    if relative_mean == 'Lower':
        print(f"F-C.I.: ({-np.inf}, {F_lower:.5f})")
    else:
        print(f"F-C.I.: ({F_upper:.5f}, {np.inf})")



# Chi - squared test

## Two Sample

In [0]:
import numpy as np
from scipy import stats

# -----------------------------
# Observed Frequencies (2x2)
# -----------------------------
#            Like  Dislike
# Male        30      10
# Female      20      20

# observed = np.array([
#     [30, 10],
#     [20, 20]
# ])
observed = np.array([
    [12, 108],
    [24, 156]
])

# -----------------------------
# Chi-Square Test
# -----------------------------
chi2_stat, p_value, df, expected = stats.chi2_contingency(observed)

# -----------------------------
# Critical Value
# -----------------------------
alpha = 0.05
chi2_critical = stats.chi2.ppf(1 - alpha, df)

# -----------------------------
# Output
# -----------------------------
print("Observed Frequencies:")
print(observed)

print("\nExpected Frequencies:")
print(expected.round(2))

print(f"\nChi-Square Statistic: {chi2_stat:.4f}")
print(f"Degrees of Freedom: {df}")
print(f"Critical Value (α = {alpha}): {chi2_critical:.4f}")
print(f"p-value: {p_value:.4f}")

# -----------------------------
# Hypothesis Decision
# -----------------------------
if chi2_stat > chi2_critical:
    print("\nReject the Null Hypothesis")
    print("Conclusion: Variables are dependent")
else:
    print("\nFail to Reject the Null Hypothesis")
    print("Conclusion: Variables are independent")


## More than Two populations

## Test for Independence